In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load variables from .env
load_dotenv()

# Access the token
hf_token = os.getenv("HF_TOKEN")
model_id = "meta-llama/Meta-Llama-3.1-8B"

c:\Users\User\Documents\TECHNION\semester 7\חיפוש מחקר\soudry\itay\k-v clustering\k_clustering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load model with original float16 weights
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16, 
    device_map="auto", # change to "cpu" for initial loading into system RAM to prevent crashing
    token=hf_token
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:18<00:00,  4.52s/it]
Some parameters are on the meta device because they were offloaded to the disk.


In [3]:
import torch

# make sure the version contains the suffix+cu121
print(f"Version: {torch.__version__}") 

# make sure returns True
print(f"Is CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

print(next(model.parameters()).device)

Version: 2.5.1+cu121
Is CUDA available: True
Using GPU: NVIDIA GeForce GTX 1660 Ti
cuda:0


In [4]:
def get_W_K(heads, w_k_name):
    num_heads, head_dim, hidden_size = heads.shape # [8, 128, 4096]

    if w_k_name == "w_k1":
        return heads[0]

    elif w_k_name == "avg W_Ki":
        return torch.mean(heads, dim=0)

    elif w_k_name == "svd":
        wk_full = heads.reshape(-1, hidden_size) 
        U, S, Vh = torch.linalg.svd(wk_full, full_matrices=False)
        
        # We take the top 'head_dim' (128) singular values and vectors.
        # This is the "best possible" matrix of size 128x4096 
        # that represents the shared information of the full 1024x4096 matrix.
        k = head_dim 
        S_k = torch.diag(S[:k])
        Vh_k = Vh[:k, :]

        # We don't need U here because we want a representative 'Head' 
        # that lives in the same row-space (V) as the original weights.
        # The reference is constructed from the top 128 principal directions.
        svd_reference = S_k @ Vh_k # Resulting shape: [128, 4096]
        
        return svd_reference

    elif w_k_name == "svd (avg)":
        avg_head = torch.mean(heads, dim=0)
        # We use full_matrices=False for efficiency
        U, S, Vh = torch.linalg.svd(avg_head, full_matrices=False)
        
        # 3. Reconstruct using only the first singular value (Rank-1 Approximation)
        # This represents the "Principal Component" of the W_K projection
        # Formula: u_1 * s_1 * v_1^T
        s_1 = S[0]
        u_1 = U[:, 0:1]
        vh_1 = Vh[0:1, :]
        
        svd_reference = u_1 @ (s_1 * vh_1)
        return svd_reference

    elif w_k_name == "I":
        return torch.eye(head_dim, hidden_size)
    
    elif w_k_name == "0":
        return torch.zeros(head_dim, hidden_size)
    else:
        raise ValueError(f"Invalid W_K name: {w_k_name}")


In [6]:
##old?
def find_optimal_A_ls_sq(W_ki_raw, W_ref_raw):
    W_ki = W_ki_raw.t()  # [4096, 128]
    W_ref = W_ref_raw.t() # [4096, 128]
    
    W_ki_pinv = torch.linalg.pinv(W_ki)
    A_opt = W_ki_pinv @ W_ref # [128, 128]

    return A_opt

def get_least_squares_distance(W_ki_raw, W_ref_raw):
    # Step 1: Transpose to work with [4096, 128]
    # Now each column is a dimension in the head space
    W_ki = W_ki_raw.t()  # [4096, 128]
    W_ref = W_ref_raw.t() # [4096, 128]
    
    # Step 2: Solve for A [128, 128] such that: W_ki @ A ≈ W_ref
    # This is a classic Least Squares problem
    # A = (W_ki^T @ W_ki)^-1 @ W_ki^T @ W_ref
    W_ki_pinv = torch.linalg.pinv(W_ki)
    A_opt = W_ki_pinv @ W_ref # [128, 128]

    #full rank check - results: A is usually of full rank, sometimes one missing
    #rank = torch.linalg.matrix_rank(A_opt)
    #print(f"Rank of A_opt: {rank.item()} (Max possible: 128)")
    
    # Step 3: Reconstruction and Distance
    W_ref_approx = W_ki @ A_opt
    
    dist_sq = torch.norm(W_ref_approx - W_ref, p='fro')**2
    ref_norm_sq = torch.norm(W_ref, p='fro')**2
    
    return (dist_sq / ref_norm_sq).item() * 100, A_opt

In [ ]:
##old?
def calculate_dist_with_optimal_A(w_k_name="svd", opt_mat_func=get_least_squares_distance):
    # Setup and Loop
    num_layers = len(model.model.layers)
    num_heads = 8
    results_matrix = np.zeros((num_layers, num_heads + 1))

    print("Calculating Optimized Procrustes Distances...")

    for layer_idx in range(num_layers):
        wk_full = model.model.layers[layer_idx].self_attn.k_proj.weight.data.to(torch.float32)
        heads = wk_full.reshape(num_heads, 128, 4096)
        
        # get reference matrix W_K
        w_k_ref = get_W_K(heads, w_k_name) 
        
        layer_dists = []
        for h_idx in range(num_heads):
            d, A = opt_mat_func(heads[h_idx], w_k_ref)
            results_matrix[layer_idx, h_idx] = d
            layer_dists.append(d)
        
        results_matrix[layer_idx, num_heads] = np.mean(layer_dists)

    # Visualization
    plt.figure(figsize=(10, 12))
    head_labels = [f"H{i}" for i in range(num_heads)] + ["AVG"]
    sns.heatmap(results_matrix, annot=True, fmt=".1f", cmap='magma', 
                xticklabels=head_labels, yticklabels=range(num_layers),
                annot_kws={"size": 7})

    plt.title(f"Llama 3.1 8B: Minimized Relative Distance (After Optimal A {opt_mat_func.__name__}, w_K {w_k_name})", fontsize=14)
    plt.show()

    print(f"Global Average Distance after alignment: {np.mean(results_matrix[:, num_heads]):.2f}%")

In [ ]:
##old?
# initial finction from gemini

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns

# 1. פונקציית עזר לשליפת אקטיבציות
class ActivationHook:
    def __init__(self):
        self.activation = None
    def __call__(self, module, input, output):
        # ב-Llama 3.1, הפלט של q_proj/k_proj הוא [batch, seq, hidden]
        self.activation = output.detach()

def compare_attention_with_A(model_id, layer_idx, head_idx, A_opt, W_k_ref):
    """
    A_opt: המטריצה שמצאת [128, 128]
    W_k_ref: מטריצת הרפרנס [128, 4096]
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
    
    # הגדרת ה-Hooks
    q_hook = ActivationHook()
    k_hook = ActivationHook()
    
    # הצמדת ה-Hooks לשכבות הפרוג'קציה
    q_proj = model.model.layers[layer_idx].self_attn.q_proj
    k_proj = model.model.layers[layer_idx].self_attn.k_proj
    handle_q = q_proj.register_forward_hook(q_hook)
    handle_k = k_proj.register_forward_hook(k_hook)

    # הרצה על טקסט דגימה
    prompt = "Understanding the alignment of attention heads in large language models."
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        model(**inputs)

    # שליפת האקטיבציות (לפני RoPE)
    # shapes: [1, seq_len, 4096]
    q_act = q_hook.activation
    k_act = k_hook.activation
    
    # חילוץ הראש הספציפי [seq_len, 128]
    # הערה: Llama 3.1 משתמש ב-GQA, יש לוודא אינדקס ראש תקין ל-K
    q_head = q_act[0].view(-1, 32, 128)[:, head_idx, :]
    k_head = k_act[0].view(-1, 8, 128)[:, head_idx % 8, :] 

    # --- חישוב מקורי (לפני RoPE) ---
    original_scores = torch.matmul(q_head, k_head.t())

    # --- חישוב מקורב עם A (לפני RoPE) ---
    # לפי הפיתוח שלך: Q_corrected = Q @ (A^-1)^T
    # נשתמש ב-pinv ליתר ביטחון נומרי
    A_corr = torch.linalg.pinv(A_opt).t().to(device).to(torch.float16)
    q_corrected = torch.matmul(q_head, A_corr)
    
    # דימוי של K_ref מהאקטיבציות (X @ W_k_ref^T)
    # נשתמש ב-Embedding של הקלט כ-X
    # לצורך הבדיקה, נשתמש ב-K שנוצר ישירות מהרפרנס
    hidden_states = model.model.embed_tokens(inputs.input_ids)
    k_ref_act = torch.matmul(hidden_states[0], W_k_ref.t().to(device).to(torch.float16))
    
    approx_scores = torch.matmul(q_corrected, k_ref_act.t())

    # --- מדידת הסטייה ---
    divergence = torch.norm(original_scores - approx_scores) / torch.norm(original_scores)
    
    print(f"Layer {layer_idx}, Head {head_idx}")
    print(f"Relative Divergence (Pre-RoPE): {divergence.item():.4%}")

    # ניקוי
    handle_q.remove()
    handle_k.remove()
    
    return original_scores, approx_scores

# דוגמה לשימוש (יש להזין את המטריצות שלך):
# scores_orig, scores_approx = compare_attention_with_A("meta-llama/Llama-3.1-8B", 0, 1, A_opt, W_k_ref)

In [ ]:
##old?
## problem with k_ref_sim (uses approx instead of real X values) so i created another vertion in the next cell

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# --- Utility: Your existing A optimization function ---
def find_optimal_A_ls_sq(W_ki_raw, W_ref_raw):
    """
    Computes the optimal linear transformation A [128, 128]
    such that (W_ki_raw.T @ A) approximates W_ref_raw.T
    """
    W_ki = W_ki_raw.t()   # [4096, 128]
    W_ref = W_ref_raw.t()  # [4096, 128]
    
    W_ki_pinv = torch.linalg.pinv(W_ki)
    A_opt = W_ki_pinv @ W_ref # [128, 128]
    return A_opt

# --- Hook Class to capture activations globally ---
class GlobalActivationHook:
    def __init__(self):
        self.q_activations = {}
        self.k_activations = {}

    def get_q_hook(self, layer_idx):
        def hook(module, input, output):
            self.q_activations[layer_idx] = output.detach()
        return hook

    def get_k_hook(self, layer_idx):
        def hook(module, input, output):
            self.k_activations[layer_idx] = output.detach()
        return hook
        

def run_comprehensive_divergence_check(model, tokenizer, prompts, w_k_name="svd"):
    """
    Analyzes divergence across ALL layers and ALL heads for a list of prompts.
    """
    device = next(model.parameters()).device
    num_layers = len(model.model.layers)
    num_heads = 8  # Llama 3.1 8B Key heads (GQA)
    q_heads = 32
    
    # 1. Register Hooks for all layers
    hooks_container = GlobalActivationHook()
    handles = []
    for i in range(num_layers):
        h_q = model.model.layers[i].self_attn.q_proj.register_forward_hook(hooks_container.get_q_hook(i))
        h_k = model.model.layers[i].self_attn.k_proj.register_forward_hook(hooks_container.get_k_hook(i))
        handles.extend([h_q, h_k])

    # Matrix to store aggregate results: [layers, heads]
    total_divergence = np.zeros((num_layers, num_heads))

    # 2. Process Prompts
    print(f"Starting analysis on {len(prompts)} prompts...")
    
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            model(**inputs)
        
        # Calculate divergence for each layer and head using captured activations
        for layer_idx in range(num_layers):
            # Get current layer weights to compute A_opt and Reference
            wk_full = model.model.layers[layer_idx].self_attn.k_proj.weight.data.to(torch.float32)
            heads_weights = wk_full.reshape(num_heads, 128, 4096)
            
            # W_ref should match the logic used in your weight analysis (e.g., SVD of the layer)
            # You must have your get_W_K function available in scope
            W_ref = get_W_K(heads_weights, w_k_name).to(device).to(torch.float16)
            
            # Get Activations for this layer
            q_act = hooks_container.q_activations[layer_idx][0] # [seq, 4096]
            k_act = hooks_container.k_activations[layer_idx][0] # [seq, 4096]
            
            for h_idx in range(num_heads):
                W_ki = heads_weights[h_idx].to(device).to(torch.float16)
                
                # Step A: Find the optimal A for this specific head/layer weight
                A_opt = find_optimal_A_ls_sq(W_ki, W_ref)
                
                # Step B: Prepare Corrected Query (A^-1)^T
                # Note: We use pinv for stability as Rank might be < 128
                A_corr = torch.linalg.pinv(A_opt).t()
                
                # Step C: Extract Activation slices
                # Map K-head index to corresponding Q-heads if necessary, 
                # here we check the 1-to-1 relationship for simplicity
                q_slice = q_act.view(-1, q_heads, 128)[:, h_idx, :]
                k_slice = k_act.view(-1, num_heads, 128)[:, h_idx, :]
                
                # Original Attention Score (Pre-RoPE)
                orig_scores = torch.matmul(q_slice, k_slice.t())
                
                # Corrected Attention Score
                # Q_corr = Q @ A_corr_T, K_ref_act = X @ W_ref_T
                # Instead of running X again, we can conceptually simulate K_ref_act 
                # if we have the input hidden states, but for weights comparison:
                q_corrected = torch.matmul(q_slice, A_corr)
                
                # Extract hidden states to compute what the reference head would have produced
                # This requires hooking the input to the layer or using embed_tokens
                # For this validation, we use the property: k_ref_act = x @ W_ref.T
                # We can derive x from k_act @ pinv(W_ki.T)
                k_ref_sim = torch.matmul(k_slice, torch.linalg.pinv(W_ki).t() @ W_ref.t())
                
                approx_scores = torch.matmul(q_corrected, k_ref_sim.t())
                
                # Divergence calculation
                div = torch.norm(orig_scores - approx_scores) / torch.norm(orig_scores)
                total_divergence[layer_idx, h_idx] += div.item()

    # Average divergence over all prompts
    total_divergence /= len(prompts)

    # 3. Cleanup Hooks
    for handle in handles:
        handle.remove()

    # 4. Visualization
    plt.figure(figsize=(12, 14))
    sns.heatmap(total_divergence, annot=True, fmt=".2%", cmap="YlGnBu")
    plt.title(f"Mean Attention Divergence across {len(prompts)} Prompts\n(Ref: {w_k_name} + Least Squares A)")
    plt.xlabel("Head Index")
    plt.ylabel("Layer Index")
    plt.show()

    return total_divergence

# --- Execution Example ---
# list_of_prompts = [
#    "The capital of France is Paris.",
#    "Deep learning is a subset of machine learning based on artificial neural networks.",
#    "To be or not to be, that is the question.",
#    "Quantum computing uses qubits to perform calculations."
# ]
# results = run_comprehensive_divergence_check(model, tokenizer, list_of_prompts)

In [ ]:
##old?
## something's not right, gemini used unimplemented functions
import torch

class ActivationHook:
    def __init__(self):
        self.input_x = None
        self.q_out = None
        self.k_out = None

    def __call__(self, module, input, output):
        # input is a tuple: (hidden_states,)
        # output is the result of the projection
        self.input_x = input[0].detach() 
        self.output = output.detach()

def run_exact_divergence_check(model, tokenizer, prompts, w_k_ref_list):
    device = next(model.parameters()).device
    num_layers = len(model.model.layers)
    num_heads = 8 
    
    results_linear = np.zeros((num_layers, num_heads))
    results_rope = np.zeros((num_layers, num_heads))

    # 1. Register Hooks for all layers on the Projection modules
    # We'll use the k_proj to capture the input X (it's the same for q_proj)
    hooks = {}
    for i in range(num_layers):
        hooks[i] = ActivationHook()
        model.model.layers[i].self_attn.k_proj.register_forward_hook(hooks[i])
        # We also need Q activations
        # (Assuming you add a similar hook for q_proj to get q_out)

    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            model(**inputs)

        for l_idx in range(num_layers):
            # The real X entering the attention layer
            X = hooks[l_idx].input_x[0] # Shape: [seq_len, 4096]
            W_K_shared = w_k_ref_list[l_idx].to(device).to(torch.float16)
            
            for h_idx in range(num_heads):
                # Get original weights and calculate A
                W_ki = model.model.layers[l_idx].self_attn.k_proj.weight.data.reshape(num_heads, 128, 4096)[h_idx] ## maybe optimize with creating "heads" out of loop
                A_opt = find_optimal_A_ls_sq(W_ki, W_K_shared)
                A_corr = torch.linalg.pinv(A_opt).t()

                # Original Q and K (Pre-RoPE) from hooks
                q_head = get_q_from_hook(l_idx, h_idx) # [seq_len, 128]
                k_head = get_k_from_hook(l_idx, h_idx) # [seq_len, 128]

                # --- Exact Ground Truth ---
                H_orig = torch.matmul(q_head, k_head.t())

                # --- Your Fixed Calculation (Using real X) ---
                # Q_fixed = Q_orig @ (A^-1)^T
                q_fixed = q_head @ A_corr
                
                # K_shared = X @ W_K_shared^T (No approximation!)
                k_shared = torch.matmul(X, W_K_shared.t())
                
                H_fixed = torch.matmul(q_fixed, k_shared.t())

                # Calculation 1: Linear Divergence (Relative Norm)
                div_lin = torch.norm(H_orig - H_fixed) / torch.norm(H_orig)
                results_linear[l_idx, h_idx] += div_lin.item()

                # --- Calculation 2: RoPE Impact ---
                # Now we apply RoPE to both and compare
                q_r_orig, k_r_orig = apply_rope(q_head, k_head, l_idx, model)
                q_r_fixed, k_r_fixed = apply_rope(q_fixed, k_shared, l_idx, model)
                
                H_rope_orig = torch.matmul(q_r_orig, k_r_orig.t())
                H_rope_fixed = torch.matmul(q_r_fixed, k_r_fixed.t())

                div_rope = torch.norm(H_rope_orig - H_rope_fixed) / torch.norm(H_rope_orig)
                results_rope[l_idx, h_idx] += div_rope.item()

    return results_linear / len(prompts), results_rope / len(prompts)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# --- Utility: Your existing A optimization function ---
def find_optimal_A_ls_sq(W_ki_raw, W_ref_raw):
    """
    Computes the optimal linear transformation A [128, 128]
    such that (W_ki_raw.T @ A) approximates W_ref_raw.T
    """
    W_ki = W_ki_raw.t().to(torch.float32)   # [4096, 128]
    W_ref = W_ref_raw.t().to(torch.float32)  # [4096, 128]
    
    W_ki_pinv = torch.linalg.pinv(W_ki)
    A_opt = W_ki_pinv @ W_ref # [128, 128]
    return A_opt.to(W_ki_raw.dtype)

# --- 1. Hook Container Class ---
class AttentionHookContainer:
    def __init__(self):
        self.q_out = {}      # Captured Q activations
        self.k_out = {}      # Captured K activations
        self.input_x = {}    # Captured real X (hidden states)

    def get_hook(self, layer_idx, name):
        def hook(module, input, output):
            if name == 'q':
                self.q_out[layer_idx] = output.detach().float()
            elif name == 'k':
                self.k_out[layer_idx] = output.detach().float()
                self.input_x[layer_idx] = input[0].detach().float() # Capture the real X
        return hook

# --- 2. Slicing Utilities (GQA Aware) ---
def get_q_from_hook(layer_idx, q_head_idx, hooks_container):
    # Llama 3.1 8B: 32 Q heads, each dim 128 (Total 4096)
    q_all = hooks_container.q_out[layer_idx]
    seq_len = q_all.shape[1]
    return q_all.view(seq_len, 32, 128)[:, q_head_idx, :]

def get_k_from_hook(layer_idx, k_head_idx, hooks_container):
    # Llama 3.1 8B: 8 K heads (GQA), each dim 128 (Total 1024)
    k_all = hooks_container.k_out[layer_idx]
    seq_len = k_all.shape[1]
    return k_all.view(seq_len, 8, 128)[:, k_head_idx, :]

# --- 3. RoPE Application (Architecture Specific) ---
def apply_rope_to_slice(q_slice, k_slice, layer_idx, model, device):
    """
    Applies the model's actual Rotary Positional Embedding logic.
    """
    rotary_emb = model.model.rotary_emb 
    seq_len = q_slice.shape[0]
    
    # Create position IDs [1, seq_len]
    position_ids = torch.arange(seq_len, device=device).unsqueeze(0)
    
    # Get cos/sin from the model's rotary_emb
    cos, sin = rotary_emb(k_slice.unsqueeze(0).transpose(1, 2), position_ids)
    
    # Apply rotation using the model's internal function
    from transformers.models.llama.modeling_llama import apply_rotary_pos_emb
    # apply_rotary_pos_emb expects [batch, num_heads, seq, dim]
    q_r, k_r = apply_rotary_pos_emb(q_slice.unsqueeze(0).unsqueeze(1), 
                                   k_slice.unsqueeze(0).unsqueeze(1), 
                                   cos, sin)
    
    return q_r.squeeze(), k_r.squeeze()

# --- 4. Main Divergence Analysis Function ---
def run_full_divergence_analysis(model, tokenizer, prompts, method="svd"):
    device = next(model.parameters()).device
    num_layers = len(model.model.layers)
    num_k_heads = 8
    
    results_linear = np.zeros((num_layers, num_k_heads))
    results_rope = np.zeros((num_layers, num_k_heads))
    
    hooks = AttentionHookContainer()
    handles = []
    
    # Register Hooks
    for i in range(num_layers):
        h_q = model.model.layers[i].self_attn.q_proj.register_forward_hook(hooks.get_hook(i, 'q'))
        h_k = model.model.layers[i].self_attn.k_proj.register_forward_hook(hooks.get_hook(i, 'k'))
        handles.extend([h_q, h_k])

    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            model(**inputs)

        for l_idx in range(num_layers):
            #wk_full = model.model.layers[l_idx].self_attn.k_proj.weight.to(device)

            """param = model.model.layers[l_idx].self_attn.k_proj.weight
            if param.is_meta:
                wk_full = param.data.to(device) 
            else:
                wk_full = param.detach().to(device)"""

            """import accelerate

            param = model.model.layers[l_idx].self_attn.k_proj.weight
            with accelerate.init_empty_weights():
                wk_full = accelerate.utils.set_module_tensor_to_device(
                    model.model.layers[l_idx].self_attn.k_proj, "weight", device=device
                )
            wk_full = model.model.layers[l_idx].self_attn.k_proj.weight.data"""

            """model.model.layers[l_idx].self_attn.to(device)
            wk_full = model.model.layers[l_idx].self_attn.k_proj.weight.data"""

            k_proj_layer = model.model.layers[l_idx].self_attn.k_proj

            """if k_proj_layer.weight.is_meta:
                from accelerate.utils import set_module_tensor_to_device
                k_proj_layer #.to(device) """

            wk_full = k_proj_layer.weight.data

            heads_weights = wk_full.view(8, 128, 4096).to(torch.float32)
            # calculated again for each prompt to save memory and not save all the W_K of each layer in the GPU, can change later
            W_K_shared = get_W_K(heads_weights, w_k_name=method)#.to(torch.float16).to(device)

            X = hooks.input_x[l_idx][0].to(torch.float32) # Real X: [seq, 4096]
            
            # Map Q heads to K heads (GQA: 4 Q heads per 1 K head)
            for k_idx in range(num_k_heads):
                # We analyze the first Q head of each K group
                q_idx = k_idx * 4 ### I think not all Qs are used, only 1/4, later implement loop to check all Qs
                
                # A. Find Optimal A for weights
                W_ki = model.model.layers[l_idx].self_attn.k_proj.weight.data.reshape(8, 128, 4096)[k_idx].to(torch.float16)
                A_opt = find_optimal_A_ls_sq(W_ki, W_K_shared) # Your function
                A_corr = torch.linalg.pinv(A_opt.to(torch.float32)).t().to(device, dtype=torch.float16)

                ###to fix - go over all Qs, 
                ###for q_sub_idx in range(4):
                ###    q_idx = k_idx * 4 + q_sub_idx
                # B. Extract Original Activations
                q_orig = get_q_from_hook(l_idx, q_idx, hooks).to(torch.float32)
                k_orig = get_k_from_hook(l_idx, k_idx, hooks).to(torch.float32)

                # --- 1. Linear Test (No RoPE) ---
                H_orig = torch.matmul(q_orig, k_orig.t())
                
                q_fixed = q_orig @ A_corr
                k_shared = X @ W_K_shared.t()
                H_fixed = torch.matmul(q_fixed, k_shared.t())
                
                div_lin = (torch.norm(H_orig - H_fixed, p='fro')**2) / (torch.norm(H_orig, p='fro')**2)
                results_linear[l_idx, k_idx] += div_lin.item()

                # --- 2. RoPE Test ---  ### make sure apply_rope_to_slice func is correct 
                q_r_orig, k_r_orig = apply_rope_to_slice(q_orig, k_orig, l_idx, model, device)
                q_r_fixed, k_r_fixed = apply_rope_to_slice(q_fixed, k_shared, l_idx, model, device)
                
                H_rope_orig = torch.matmul(q_r_orig, k_r_orig.t())
                H_rope_fixed = torch.matmul(q_r_fixed, k_r_fixed.t())

                div_rope = (torch.norm(H_rope_orig - H_rope_fixed, p='fro')**2) / (torch.norm(H_rope_orig, p='fro')**2)
                results_rope[l_idx, k_idx] += div_rope.item()

    # Cleanup
    for h in handles: h.remove()
    
    return results_linear / len(prompts), results_rope / len(prompts)

In [ ]:
##old?
def prepare_w_k_ref_list(model, method="svd"):
    """
    Iterates through all layers of the model and generates a shared W_K reference
    per layer using the provided get_W_K function.
    """
    device = next(model.parameters()).device
    num_layers = len(model.model.layers)
    w_k_ref_list = []
    
    print(f"Generating shared W_K references for {num_layers} layers using method: {method}...")

    for i in range(num_layers):
        # 1. Extract the full K_proj weight matrix
        # Shape: [1024, 4096] (for Llama 3.1 8B)
        wk_full = model.model.layers[i].self_attn.k_proj.weight.data.detach()
        
        # 2. Reshape to separate the 8 Key heads
        # Shape: [8, 128, 4096]
        heads_weights = wk_full.view(8, 128, 4096)
        
        # 3. Generate the shared reference matrix A for this layer
        # Output: [128, 4096]
        W_K_shared = get_W_K(heads_weights, w_k_name=method)
        
        # Store in the list (ensuring it stays on the correct device and dtype)
        w_k_ref_list.append(W_K_shared.to(device).to(torch.float16))
        
    print("Reference list prepared successfully.")
    return w_k_ref_list

def prepare_w_k_ref_list(model, method="svd"):
    device = next(model.parameters()).device
    num_layers = len(model.model.layers)
    w_k_ref_list = []
    
    print(f"Generating references for {num_layers} layers...")

    for i in range(num_layers):
        # 1. Extract weights
        wk_full = model.model.layers[i].self_attn.k_proj.weight.data.detach()
        
        # 2. Reshape
        heads_weights = wk_full.view(8, 128, 4096)
        
        # --- תיקון השגיאה ---
        # מעבירים ל-float32 לפני שליחת הנתונים ל-get_W_K (שמשתמשת ב-SVD)
        heads_weights_f32 = heads_weights.to(torch.float32)
        
        # 3. Generate shared reference
        W_K_shared_f32 = get_W_K(heads_weights_f32, w_k_name=method)
        
        # מחזירים ל-float16 (Half) כדי שיתאים לשאר המודל ב-GPU
        W_K_shared = W_K_shared_f32.to(torch.float16)
        # ---------------------
        
        w_k_ref_list.append(W_K_shared.to(device))
        
    return w_k_ref_list

In [14]:
# 1. Define a list of common/standard prompts
common_prompts = [
    #"The capital of France is Paris and the Eiffel Tower is located there.",
    "Solve the following math problem: If I have 3 apples and you give me 5 more, how many do I have?",
    #"Write a short Python function to sort a list of numbers."
]

# 2. Prepare your reference matrices (This is a placeholder for your W_K matrices)
# You should have one [128, 4096] matrix per layer.
# Example: w_k_ref_list = [layer_0_ref, layer_1_ref, ..., layer_31_ref]
#w_k_ref_list = prepare_w_k_ref_list(model, method="svd")


tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# 3. Run the analysis
results_lin, results_rope = run_full_divergence_analysis(
    model=model, 
    tokenizer=tokenizer, 
    prompts=common_prompts, 
    method="svd"
)

# 4. Print a quick summary of the first layer
print(f"Layer 0 Linear Divergence: {results_lin[0].mean():.2%}")
print(f"Layer 0 RoPE Divergence: {results_rope[0].mean():.2%}")

RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != struct c10::Half

In [ ]:
def plot_comparison(res_lin, res_rope):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    sns.heatmap(res_lin, annot=False, cmap="YlGnBu", ax=ax1)
    ax1.set_title("Linear Divergence (Pre-RoPE)\nGoal: Close to 0%")
    ax1.set_xlabel("Key Head Index")
    ax1.set_ylabel("Layer Index")

    sns.heatmap(res_rope, annot=False, cmap="OrRd", ax=ax2)
    ax2.set_title("Full Divergence (With RoPE)\nImpact of Positional Encoding")
    ax2.set_xlabel("Key Head Index")
    
    plt.tight_layout()
    plt.show()

plot_comparison(results_lin, results_rope)